# Lab 1 — HRM 데이터 모델링 (60분)
**목표:** 기기 마스터·원본 이력·최신 상태의 역할을 구분하고 Entity별 복합키를 보존합니다.
기존 lab1의 기기 3건, 이벤트 18건, 상태 12건을 사용합니다. 이벤트는 Entity A만 포함하며
상태의 B~F는 별도 교육용 구조 예시입니다. anomaly_results는 빈 배열입니다.
UUID 문자열과 ObjectId를 임의로 같은 타입으로 바꾸지 않습니다. MODE 코드를 문 열림 의미로 추정하지 않습니다.

위에서 아래로 실행하세요. 전용 DB `hrm_workshop_lab1`을 사용합니다.
흐름: 구조·타입(10분) → 키·조회(15분) → 최신 상태(10분) → 부분 갱신(15분) → 모델 비교(10분).

## 공유 DB 실행 역할 — 먼저 확인하세요

| 표시 | 실행 담당 | 작업 |
|---|---|---|
| 🔴 강사 단독 실행 | 지정 강사 1명 | 적재·인덱스/컬렉션 생성·상태/알림 변경·임베딩 갱신 |
| 🟢 다중 실행 가능 | 모든 참가자 | find/aggregate/explain·건수/타입 확인·로컬 계산 |

**참가자:** 저장소 루트의 `Shared_DB_Participant.ipynb`에서 해당 Lab을 실행하세요.
이 파일의 강사 셀을 생략하면 변수 정의가 빠질 수 있으므로 참가자용은 독립된 읽기 코드로 구성했습니다.
**강사:** 아래 `INSTRUCTOR_MODE = True`로 변경한 뒤 준비/시연하세요. 참가자는 False를 유지합니다.
기본값 False에서는 쓰기 셀이 DB 변경 전에 중단됩니다. 표시된 이전 출력은 강사의 실행 예시입니다.

멱등 upsert도 DB 쓰기이므로 강사 전용입니다. "1회"는 **여러 사람이 실행하지 않는다는 의미**입니다.
재전송·부분 갱신·timer 테스트는 강사 한 명이 시나리오 순서대로 반복합니다.
Time Series insertMany·TTL insert·Kafka 재발행은 반복 시 중복/만료 변화가 발생할 수 있습니다.
공유 데이터 준비가 끝나면 강사가 완료를 안내하고 참가자 조회를 시작합니다.


In [ ]:
# 참가자는 False 유지. 지정 강사 한 명만 True로 변경합니다.
INSTRUCTOR_MODE = False

### 🔴 강사 단독 실행 · DB 변경
강사 준비·시연 단계에서만 실행합니다. 참가자 동시 실행 및 공유 DB Run All 금지.

In [1]:
# [강사 단독 실행] 공유 DB 변경은 지정된 강사 한 명만 수행합니다.
if not INSTRUCTOR_MODE:
    raise RuntimeError("강사 전용 셀입니다. 참가자는 Shared_DB_Participant.ipynb를 실행하세요.")
# 프로젝트 공통 모듈을 불러옵니다. 인증정보는 로컬 파일/환경변수/숨김 입력에서 읽습니다.
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
from copy import deepcopy
from collections import Counter
from IPython.display import display
from bson import json_util
import pandas as pd
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab2/events.ejson").exists())
sys.path.insert(0, str(root))
from workshop_utils import connect, load_samples, insert_once, table
UTC = timezone.utc

# Extended JSON을 BSON 타입에 대응하는 Python 값으로 읽습니다.
samples = {p.stem: json_util.loads(p.read_text(), json_options=json_util.JSONOptions(tz_aware=True))
           for p in (root / "lab1").glob("*.ejson")}
assert {k: len(v) for k, v in samples.items()} == {
    "devices": 3, "device_events": 18, "device_current_state": 12, "anomaly_results": 0}
client, db = connect("hrm_workshop_lab1")
for name, documents in samples.items():
    insert_once(db[name], documents)
# 빈 anomaly_results도 역할을 보여주기 위해 일반 컬렉션으로 생성합니다.
if "anomaly_results" not in db.list_collection_names():
    db.create_collection("anomaly_results")
display(table([{"collection": name, "sampleCount": len(docs), "storedCount": db[name].count_documents({})}
               for name, docs in samples.items()]))
display(table(samples["device_events"][:1]))

연결 성공 / MongoDB 8.0.34 / DB hrm_workshop_lab1


,collection,sampleCount,storedCount
0,anomaly_results,0,0
1,device_events,18,18
2,device_current_state,12,12
3,devices,3,3


,_id,deviceId,locationId,eventTime,receivedAt,schemaVersion,resource.href,resource.unitId,resource.component,notification.source,...,context.prdYear,context.prdMonth,context.softwareVersion,context.newVersionAvailable,context.deviceCategoryCode,context.regionCode,payload.resourceTypes,payload.interfaces,payload.supportedModes,payload.modes
0,a4dcb3e7-5e69-5c4f-87bf-cc55d91d2f7d,1ecce622-a408-5579-bbef-56d92499a46a,1dceb7c9-affd-511d-afac-bf54266f2a1a,2026-10-12 08:03:00+00:00,2026-10-12 08:03:00.300000+00:00,1,/resourceA/vs/0,0,main,DEVICE,...,2020,10,00000000.000000,0,A0,DEMO-SEOUL,[resourceType.A],[interface.a],"[SUPPORTED_MODE_01, SUPPORTED_MODE_02, SUPPORT...","[MODE_01, MODE_02, MODE_03, MODE_04, MODE_05, ..."


## 1. 역할과 타입
| 컬렉션 | 역할 | 대표 접근 |
|---|---|---|
| devices | 상대적으로 안정적인 기기 정보 | 기기 ID 조회 |
| device_events | 변경 이력 | 기기·리소스·기간 조회 |
| device_current_state | Entity별 최신 상태 | deviceId + stateKey 조회 |
| anomaly_results | 규칙 평가 결과 | 상태·감지 시각 조회 (Lab 3) |

무한히 늘어나는 이력 배열은 기기 문서에 계속 붙이지 않습니다.
bounded modes 배열은 상태 안에 보관하고 모델 정보를 이력에 함께 저장하면 당시 모델의 관측값을 보존할 수 있습니다.

### 🟢 다중 실행 가능 · 조회/로컬 계산
DB를 변경하지 않습니다. 이 노트북의 앞 셀 변수가 필요하면 참가자 전용 노트북을 사용하세요.

In [2]:
# 동일 필드의 BSON 타입을 서버 집계로 검증합니다.
for name, field in (("devices", "_id"), ("device_events", "eventTime"),
                    ("device_current_state", "_id"), ("device_current_state", "state.modes")):
    result = list(db[name].aggregate([
        {"$group": {"_id": {"$type": "$" + field}, "count": {"$sum": 1}}}, {"$sort": {"_id": 1}}]))
    print(name, field)
    display(table(result))
assert all(isinstance(e["eventTime"], datetime) for e in samples["device_events"])

devices _id


,_id,count
0,string,3


device_events eventTime


,_id,count
0,date,18


device_current_state _id


,_id,count
0,objectId,12


device_current_state state.modes


,_id,count
0,array,6
1,missing,6


## 2. Entity별 원래 복합키를 stateKey로 표현
A/D: unitId, B: unitId+component+filterName, C: subDevice,
E: unitId+id, F: unitId+target입니다. 이 샘플에는 G/H가 없습니다.
stateKey는 정해진 순서의 JSON 배열 문자열이며 식별자의 자료형·순서를 보존합니다.

### 🔴 강사 단독 실행 · DB 변경
강사 준비·시연 단계에서만 실행합니다. 참가자 동시 실행 및 공유 DB Run All 금지.

In [3]:
# [강사 단독 실행] 공유 DB 변경은 지정된 강사 한 명만 수행합니다.
if not INSTRUCTOR_MODE:
    raise RuntimeError("강사 전용 셀입니다. 참가자는 Shared_DB_Participant.ipynb를 실행하세요.")
# 키 계약에 따라 stateKey를 재계산하고 제공된 샘플과 비교합니다.
import json
key_fields = {"A": ["unitId"], "B": ["unitId", "component", "filterName"],
              "C": ["subDevice"], "D": ["unitId"], "E": ["unitId", "id"], "F": ["unitId", "target"]}
for state in samples["device_current_state"]:
    values = [state["entity"], *[state["key"][f] for f in key_fields[state["entity"]]]]
    assert state["stateKey"] == json.dumps(values, separators=(",", ":"))
states = db.device_current_state
states.create_index([("deviceId", 1), ("stateKey", 1)], unique=True, name="device_state_unique")
D = samples["devices"][0]["_id"]
identity = {"deviceId": D, "stateKey": '["A","0"]'}
display(states.find_one(identity))
display(table(states.find({"entity": "A", "state.modes": "MODE_01"}, {"state.supportedModes": 0})))
assert states.count_documents({"entity": "A", "state.modes": "MODE_01"}) == 6

{'_id': ObjectId('000000000000000000000001'),
 'deviceId': '1ecce622-a408-5579-bbef-56d92499a46a',
 'entity': 'A',
 'key': {'unitId': '0'},
 'lastEventId': 'c7c13a04-bcbe-51bd-bc74-8f8e02fdf1eb',
 'schemaVersion': 1,
 'state': {'resourceTypes': ['resourceType.A'],
  'interfaces': ['interface.a'],
  'supportedModes': ['SUPPORTED_MODE_01',
   'SUPPORTED_MODE_02',
   'SUPPORTED_MODE_03',
   'SUPPORTED_MODE_04',
   'SUPPORTED_MODE_05',
   'SUPPORTED_MODE_06',
   'SUPPORTED_MODE_07',
   'SUPPORTED_MODE_08',
   'SUPPORTED_MODE_09',
   'SUPPORTED_MODE_10',
   'SUPPORTED_MODE_11',
   'SUPPORTED_MODE_12',
   'SUPPORTED_MODE_13',
   'SUPPORTED_MODE_14'],
  'modes': ['MODE_01', 'MODE_02', 'MODE_03', 'MODE_04', 'MODE_05', 'MODE_06']},
 'stateKey': '["A","0"]',
 'updatedAt': datetime.datetime(2026, 10, 12, 10, 3, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC'))}

,_id,deviceId,entity,lastEventId,schemaVersion,stateKey,updatedAt,key.unitId,state.resourceTypes,state.interfaces,state.modes
0,000000000000000000000001,1ecce622-a408-5579-bbef-56d92499a46a,A,c7c13a04-bcbe-51bd-bc74-8f8e02fdf1eb,1,"[""A"",""0""]",2026-10-12 10:03:00+00:00,0,[resourceType.A],[interface.a],"[MODE_01, MODE_02, MODE_03, MODE_04, MODE_05, ..."
1,000000000000000000000002,1ecce622-a408-5579-bbef-56d92499a46a,A,4e2ae2af-29c4-5d30-9418-21b829e5640d,1,"[""A"",""1""]",2026-10-12 10:04:00+00:00,1,[resourceType.A],[interface.a],"[MODE_01, MODE_02, MODE_03, MODE_04, MODE_05, ..."
2,000000000000000000000003,569489de-a3b8-5c18-ba01-7cb67d532779,A,55a19a89-2451-5252-8bfd-7584a3052432,1,"[""A"",""0""]",2026-10-12 10:06:00+00:00,0,[resourceType.A],[interface.a],"[MODE_01, MODE_02, MODE_03, MODE_04, MODE_05, ..."
3,000000000000000000000004,569489de-a3b8-5c18-ba01-7cb67d532779,A,348b616b-3e76-5a21-b3fd-9ea0dd99319e,1,"[""A"",""1""]",2026-10-12 10:07:00+00:00,1,[resourceType.A],[interface.a],"[MODE_01, MODE_02, MODE_03, MODE_04, MODE_05, ..."
4,000000000000000000000005,a5c291fd-bf34-534e-a10e-ebd5684a6504,A,cd033691-218c-56d9-b329-f489f0fdac11,1,"[""A"",""0""]",2026-10-12 10:09:00+00:00,0,[resourceType.A],[interface.a],"[MODE_01, MODE_02, MODE_03, MODE_04, MODE_05, ..."
5,000000000000000000000006,a5c291fd-bf34-534e-a10e-ebd5684a6504,A,3793d637-6134-5e05-bf37-3842e026e5fe,1,"[""A"",""1""]",2026-10-12 10:10:00+00:00,1,[resourceType.A],[interface.a],"[MODE_01, MODE_02, MODE_03, MODE_04, MODE_05, ..."


## 3. 원본 이력에서 최신 상태 계산
A의 full snapshot 이벤트만 대상으로 기기+unitId별 마지막 값을 구합니다.
부분 payload나 동일 eventTime의 충돌은 다른 계약이 필요하며 여기서는 full snapshot을 전제로 합니다.

### 🔴 강사 단독 실행 · DB 변경
강사 준비·시연 단계에서만 실행합니다. 참가자 동시 실행 및 공유 DB Run All 금지.

In [4]:
# [강사 단독 실행] 공유 DB 변경은 지정된 강사 한 명만 수행합니다.
if not INSTRUCTOR_MODE:
    raise RuntimeError("강사 전용 셀입니다. 참가자는 Shared_DB_Participant.ipynb를 실행하세요.")
# sort 후 group의 first로 리소스별 최신 이벤트를 선택합니다.
history = db.device_events
history.create_index([("deviceId", 1), ("resource.href", 1), ("eventTime", -1)], name="device_resource_time")
latest = list(history.aggregate([
    {"$sort": {"eventTime": -1, "_id": -1}},
    {"$group": {"_id": {"deviceId": "$deviceId", "unitId": "$resource.unitId"}, "event": {"$first": "$$ROOT"}}}
]))
assert len(latest) == 6
checks = []
for item in latest:
    event = item["event"]
    key = json.dumps(["A", event["resource"]["unitId"]], separators=(",", ":"))
    state = states.find_one({"deviceId": event["deviceId"], "stateKey": key})
    assert state["lastEventId"] == event["_id"] and state["state"] == event["payload"]
    assert state["updatedAt"] == event["eventTime"]
    checks.append({"deviceId": event["deviceId"], "stateKey": key, "latestEventId": event["_id"], "match": True})
display(table(checks))

,deviceId,stateKey,latestEventId,match
0,569489de-a3b8-5c18-ba01-7cb67d532779,"[""A"",""1""]",348b616b-3e76-5a21-b3fd-9ea0dd99319e,True
1,1ecce622-a408-5579-bbef-56d92499a46a,"[""A"",""0""]",c7c13a04-bcbe-51bd-bc74-8f8e02fdf1eb,True
2,a5c291fd-bf34-534e-a10e-ebd5684a6504,"[""A"",""0""]",cd033691-218c-56d9-b329-f489f0fdac11,True
3,a5c291fd-bf34-534e-a10e-ebd5684a6504,"[""A"",""1""]",3793d637-6134-5e05-bf37-3842e026e5fe,True
4,569489de-a3b8-5c18-ba01-7cb67d532779,"[""A"",""0""]",55a19a89-2451-5252-8bfd-7584a3052432,True
5,1ecce622-a408-5579-bbef-56d92499a46a,"[""A"",""1""]",4e2ae2af-29c4-5d30-9418-21b829e5640d,True


## 4. 부분 갱신과 누락·null·빈 배열
`$set: {"state.modes": ...}`는 형제 필드를 유지합니다. `$set: {"state": ...}`는 state 전체를 교체합니다.
누락은 "변경하지 않음", null은 "값 없음", []는 "빈 목록"으로 이 실습에서 정의합니다.
업무 계약마다 다를 수 있으므로 수집 시스템과 합의해야 합니다.
실습은 원본 상태를 복사한 `modeling_demo` 컬렉션에서 수행합니다.

### 🔴 강사 단독 실행 · DB 변경
강사 준비·시연 단계에서만 실행합니다. 참가자 동시 실행 및 공유 DB Run All 금지.

In [5]:
# [강사 단독 실행] 공유 DB 변경은 지정된 강사 한 명만 수행합니다.
if not INSTRUCTOR_MODE:
    raise RuntimeError("강사 전용 셀입니다. 참가자는 Shared_DB_Participant.ipynb를 실행하세요.")
# 복사한 한 문서에서 연산 차이를 실제로 검증합니다. 원본 상태 컬렉션은 수정하지 않습니다.
original = states.find_one(identity)
demo = db.modeling_demo
working = deepcopy(original)
working["_id"] = "lab1-partial-update"
demo.replace_one({"_id": working["_id"]}, working, upsert=True)
result = demo.update_one({"_id": working["_id"]}, {"$set": {"state.modes": ["LAB1-DEMO"]}}, upsert=False)
assert result.matched_count == 1
assert demo.find_one({"_id": working["_id"]})["state"]["supportedModes"] == original["state"]["supportedModes"]
# 없는 문서는 UPDATE만으로 생성하지 않습니다.
assert demo.update_one({"_id": "not-present"}, {"$set": {"state.modes": []}}, upsert=False).matched_count == 0
observed = []
for label, change in (("missing", {"$unset": {"state.modes": ""}}),
                      ("null", {"$set": {"state.modes": None}}),
                      ("empty array", {"$set": {"state.modes": []}})):
    demo.update_one({"_id": working["_id"]}, change)
    row = list(demo.aggregate([{ "$match": {"_id": working["_id"]}},
                              {"$project": {"_id": 0, "type": {"$type": "$state.modes"}}}]))[0]
    observed.append({"case": label, "bsonType": row["type"]})
assert [r["bsonType"] for r in observed] == ["missing", "null", "array"]
display(table(observed))
# 다음 실행도 같은 기준에서 시작하도록 복사본을 원래 값으로 되돌립니다.
demo.replace_one({"_id": working["_id"]}, working)

,case,bsonType
0,missing,missing
1,null,null
2,empty array,array


UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000002'), 'opTime': {'ts': Timestamp(1791950941, 1), 't': 2}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1791950941, 1), 'signature': {'hash': b'qq\xe5\xcd\xd4\x12\x0e)\xf6#+M\xff\x11O\x8a\xf5\x9c\xf9\xd2', 'keyId': 7696336607065931781}}, 'operationTime': Timestamp(1791950941, 1), 'updatedExisting': True}, acknowledged=True)

### 🟢 다중 실행 가능 · 조회/로컬 계산
DB를 변경하지 않습니다. 이 노트북의 앞 셀 변수가 필요하면 참가자 전용 노트북을 사용하세요.

In [6]:
# 필요한 기기 메타데이터만 lookup하고 문서 크기를 측정합니다.
joined = list(states.aggregate([
    {"$match": {"deviceId": D}},
    {"$lookup": {"from": "devices", "localField": "deviceId", "foreignField": "_id", "as": "device"}},
    {"$project": {"stateKey": 1, "entity": 1, "modelCode": {"$first": "$device.product.modelCode"},
                  "bsonBytes": {"$bsonSize": "$$ROOT"}}}
]))
display(table(joined))
assert all(r["modelCode"] == samples["devices"][0]["product"]["modelCode"] for r in joined)
client.close()  # 단일 노트북에서 재사용한 연결을 종료합니다.

,_id,entity,stateKey,modelCode,bsonBytes
0,000000000000000000000001,A,"[""A"",""0""]",DEMO-MODEL-1,1184
1,000000000000000000000002,A,"[""A"",""1""]",DEMO-MODEL-1,1184
2,000000000000000000000007,B,"[""B"",""0"",""main"",""DEMO-FILTER-1""]",DEMO-MODEL-1,699
3,000000000000000000000008,B,"[""B"",""0"",""main"",""DEMO-FILTER-2""]",DEMO-MODEL-1,699
4,000000000000000000000009,C,"[""C"",""DEMO-SUB-1""]",DEMO-MODEL-1,647
5,00000000000000000000000a,D,"[""D"",""0""]",DEMO-MODEL-1,626
6,00000000000000000000000b,E,"[""E"",""0"",""DEMO-INFO-1""]",DEMO-MODEL-1,660
7,00000000000000000000000c,F,"[""F"",""0"",""DEMO-FW-1""]",DEMO-MODEL-1,660


## 리뷰
- 자주 함께 조회되는 기기 메타데이터는 내장할까요, 참조할까요?
- 이력과 최신 상태의 쓰기·보존·조회 패턴이 다른 이유를 설명하세요.
- unique 키 충돌 시 원본 키 계약과 기존 중복을 먼저 확인하세요.
- 이벤트/상태 간 여러 문서 쓰기의 원자성·재시작 처리는 Lab 3에서 다룹니다.

[데이터 모델링](https://www.mongodb.com/docs/manual/data-modeling/) ·
[부분 갱신](https://www.mongodb.com/docs/manual/reference/method/db.collection.updateOne/)